# 1. LMDB to PyTorch Chunks

This notebook converts index data from the LMDB files into chunked PyTorch `.pt` files (`edge_index`, `node_features`).
This chunked format makes loading large graphs feasible on systems with limited RAM.

In [ ]:
# Configuration
import os

# EXTERNAL DRIVE CONFIGURATION
ROOT_DIR = "/Volumes/Backup Plus/Zaman/graph"
LMDB_EDGE_DIR = os.path.join(ROOT_DIR, "lmdb_edge_indexing")
LMDB_NODE_DIR = os.path.join(ROOT_DIR, "lmdb_node_mapping")
OUTPUT_CHUNKS_DIR = os.path.join(ROOT_DIR, "processed_data", "chunks")

# Chunking Parameters
BATCH_SIZE = 1_000_000
NUM_WORKERS = 4
MAX_QUEUE_SIZE = 8
SEPARATOR = ","

os.makedirs(OUTPUT_CHUNKS_DIR, exist_ok=True)

In [ ]:
# Imports
import lmdb
import torch
import numpy as np
from tqdm.notebook import tqdm
from concurrent.futures import ThreadPoolExecutor
import msgpack
import pickle
import json

In [ ]:
# Utilities
def check_futures(futures):
    """Check for exceptions in background threads."""
    for f in futures:
        if f.done() and f.exception():
            raise f.exception()

def finalize_futures(futures):
    """Wait for all futures to complete."""
    for f in futures:
        f.result()

In [ ]:
# Edge Converter
def lmdb_to_pt_chunks(
    lmdb_path: str,
    outdir: str,
    batch_size: int = BATCH_SIZE,
    workers: int = NUM_WORKERS,
    max_queue: int = MAX_QUEUE_SIZE,
    separator: str = SEPARATOR
):
    """
    Reads edge indices from LMDB and saves them as chunked PyTorch tensors.
    """
    if not os.path.exists(lmdb_path):
        print(f"LMDB not found: {lmdb_path}")
        return

    os.makedirs(outdir, exist_ok=True)
    env = lmdb.open(lmdb_path, readonly=True, lock=False, readahead=False)
    
    futures = []
    pool = ThreadPoolExecutor(max_workers=workers)
    
    print(f"Processing {lmdb_path}...")
    
    with env.begin() as txn:
        cursor = txn.cursor()
        batch_src = []
        batch_dst = []
        chunk_idx = 0
        count = 0

        def save_chunk(c_idx, src_data, dst_data):
            src_t = torch.tensor(src_data, dtype=torch.long)
            dst_t = torch.tensor(dst_data, dtype=torch.long)
            edge_index = torch.stack([src_t, dst_t], dim=0)
            torch.save(edge_index, os.path.join(outdir, f"chunk_{c_idx}.pt"))

        pbar = tqdm()
        for k, v in cursor:
            # Basic progress monitoring
            if count % 10000 == 0:
                pbar.update(10000)
                check_futures(futures)
                # Clean up completed futures to avoid memory leaks
                futures = [f for f in futures if not f.done()]

            val_str = v.decode()
            parts = val_str.split(separator)
            
            if len(parts) != 2:
                continue
                
            batch_src.append(int(parts[0]))
            batch_dst.append(int(parts[1]))
            count += 1

            if len(batch_src) >= batch_size:
                # Submit tasks if queue is not full
                while len(futures) >= max_queue:
                     # Wait for at least one to finish
                    completed = False
                    for i, f in enumerate(futures):
                        if f.done():
                            f.result() # raising exception if any
                            del futures[i]
                            completed = True
                            break
                    if not completed:
                        futures[0].result() # Force wait
                        del futures[0]

                futures.append(pool.submit(save_chunk, chunk_idx, list(batch_src), list(batch_dst)))
                batch_src = []
                batch_dst = []
                chunk_idx += 1

        # Final chunk
        if batch_src:
            futures.append(pool.submit(save_chunk, chunk_idx, batch_src, batch_dst))
            
        finalize_futures(futures)
        pool.shutdown()
        pbar.close()
        print(f"Finished processing {count} edges into {chunk_idx + 1} chunks.")

In [ ]:
# Run for all edge files found in LMDB_EDGE_DIR
# You can filter specific files if needed
files = [f for f in os.listdir(LMDB_EDGE_DIR) if f.endswith(".lmdb")]
print(f"Found edge LMDBs: {files}")

for f in files:
    name = f.replace(".lmdb", "")
    lmdb_path = os.path.join(LMDB_EDGE_DIR, f)
    out_path = os.path.join(OUTPUT_CHUNKS_DIR, f"edges_{name}")
    lmdb_to_pt_chunks(lmdb_path, out_path)

In [ ]:
# Node Features Converter (Optional - if you have features in LMDB)
# This function handles basic feature extraction if your node mapping LMDB contains more than just ID.

def decode_value(val_bytes):
    try:
        return msgpack.unpackb(val_bytes)
    except:
        try:
            return pickle.loads(val_bytes)
        except:
            try:
                return json.loads(val_bytes)
            except:
                return val_bytes.decode()

def lmdb_node_to_pt_chunks(
    lmdb_path: str,
    outdir: str,
    batch_size: int = 200_000,
    workers: int = NUM_WORKERS,
    max_queue: int = MAX_QUEUE_SIZE,
):
    if not os.path.exists(lmdb_path):
        return

    os.makedirs(outdir, exist_ok=True)
    env = lmdb.open(lmdb_path, readonly=True, lock=False)
    
    futures = []
    pool = ThreadPoolExecutor(max_workers=workers)
    print(f"Processing nodes {lmdb_path}...")

    with env.begin() as txn:
        cursor = txn.cursor()
        batch_feat = []
        chunk_idx = 0
        
        # NOTE: This assumes keys are ordered (0, 1, 2...) or you handle ID mapping logic here.
        # If keys are raw IDs, you rely on the fact that we created the mapping 0..N-1 previously.
        # This default function assumes we are just reading contiguous data.
        
        def save_node_chunk(c_idx, data_list):
            # Assuming data_list is list of feature vectors or similar
            # If data is just IDs, this might not be needed.
            # Placeholder for feature tensor creation
            # t = torch.tensor(data_list, dtype=torch.float)
            # torch.save(t, os.path.join(outdir, f"chunk_{c_idx}.pt"))
            pass

        for k, v in tqdm(cursor):
            # logic to parse node features would go here
            pass
            
    pool.shutdown()